# VISUALIZACIÓN WAVELETS

In [ ]:
import pandas as pd
import numpy as np
import pywt
import os
import matplotlib.pyplot as plt
import sys

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================

# [CONFIGURACIÓN DE USUARIO]: Define las rutas de tu entorno
BASE_DIR = " "
INPUT_DIR = os.path.join(BASE_DIR, "normalizacion")
OUTPUT_ROOT = os.path.join(BASE_DIR, "figuras_wavelet_usuarios")

# Wavelets a procesar
WAVELETS_DWT = ['haar', 'db2']  # Discretas
WAVELET_CWT = 'mexh'            # Continua

# [CONFIGURACIÓN DE USUARIO]: Límite de IPs a graficar por etiqueta para no saturar el disco. 
# Pon 'None' para procesar TODAS las IPs del dataset.
LIMIT_IPS = 10 

# Niveles máximos de descomposición para mostrar en la gráfica
MAX_LEVEL_PLOT = 8

# ==========================================
# 2. FUNCIONES DE GENERACIÓN (GRÁFICAS)
# ==========================================

def plot_mra_dwt(serie, wavelet, ip, etiqueta, output_folder):
    """Genera un PDF con la descomposición (Aproximaciones y Detalles) para DWT."""
    mode = 'per'
    coeffs = pywt.wavedec(serie, wavelet, mode=mode)
    n_levels = len(coeffs) - 1
    levels_to_plot = min(n_levels, MAX_LEVEL_PLOT)
    
    fig, axes = plt.subplots(levels_to_plot, 2, figsize=(15, 2 * levels_to_plot), sharex=True)
    fig.suptitle(f"MRA Decomposition ({wavelet.upper()}) - {etiqueta} - IP: {ip}", fontsize=16)

    for i in range(levels_to_plot):
        level_idx = i + 1
        
        # Reconstrucción del Detalle (D)
        coeffs_rec_d = [np.zeros_like(c) for c in coeffs]
        coeffs_rec_d[-level_idx] = coeffs[-level_idx]
        detail_signal = pywt.waverec(coeffs_rec_d, wavelet, mode=mode)
        if len(detail_signal) > len(serie): detail_signal = detail_signal[:len(serie)]

        # Reconstrucción de la Aproximación (A)
        coeffs_at_j = pywt.wavedec(serie, wavelet, mode=mode, level=level_idx)
        coeffs_rec_a = [np.zeros_like(c) for c in coeffs_at_j]
        coeffs_rec_a[0] = coeffs_at_j[0]
        approx_signal = pywt.waverec(coeffs_rec_a, wavelet, mode=mode)
        if len(approx_signal) > len(serie): approx_signal = approx_signal[:len(serie)]

        # Plot Aproximación
        ax_a = axes[i, 0]
        ax_a.plot(approx_signal, 'b-', linewidth=1)
        ax_a.set_ylabel(f"A_{level_idx}", fontsize=10, rotation=0, labelpad=20)
        ax_a.grid(True, linestyle=':', alpha=0.6)
        if i == 0: ax_a.set_title("Approximation (Coarse)", fontsize=12)

        # Plot Detalle
        ax_d = axes[i, 1]
        ax_d.plot(detail_signal, 'k-', linewidth=0.8)
        ax_d.set_ylabel(f"D_{level_idx}", fontsize=10, rotation=0, labelpad=20)
        ax_d.grid(True, linestyle=':', alpha=0.6)
        if i == 0: ax_d.set_title("Detail (High Freq)", fontsize=12)

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.savefig(os.path.join(output_folder, f"{etiqueta}_{ip}.pdf"))
    plt.close()

def plot_cwt_mexh(serie, ip, etiqueta, output_folder):
    """Genera un PDF con las escalas continuas para CWT (Mexican Hat)."""
    scales = np.arange(1, 15)
    coeffs, _ = pywt.cwt(serie, scales, WAVELET_CWT)
    n_scales = len(scales)
    
    fig, axes = plt.subplots(n_scales, 1, figsize=(10, 1.5 * n_scales), sharex=True)
    fig.suptitle(f"CWT Analysis ({WAVELET_CWT.upper()}) - {etiqueta} - IP: {ip}", fontsize=16)
    
    for i, ax in enumerate(axes):
        ax.plot(coeffs[i], 'r-', linewidth=0.8)
        ax.set_ylabel(f"S{scales[i]}", fontsize=9, rotation=0, labelpad=20)
        ax.grid(True, linestyle=':', alpha=0.6)
    
    axes[0].set_title("Wavelet Coefficients by Scale (Mexican Hat)", fontsize=12)
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.savefig(os.path.join(output_folder, f"{etiqueta}_{ip}.pdf"))
    plt.close()

# ==========================================
# 3. PROCESAMIENTO PRINCIPAL
# ==========================================

def generar_graficas():
    print("=== GENERACIÓN DE GRÁFICAS WAVELET (PDFs) ===")
    print(f"Directorio Salida: {OUTPUT_ROOT}")
    
    if not os.path.exists(INPUT_DIR):
        print(f"[ERROR] No existe el directorio de entrada: {INPUT_DIR}")
        return

    archivos_csv = [f for f in os.listdir(INPUT_DIR) if f.endswith(".csv")]
    
    for archivo in archivos_csv:
        etiqueta = archivo.replace(".csv", "")
        
        try:
            df = pd.read_csv(os.path.join(INPUT_DIR, archivo), index_col=0)
            
            # Seleccionar IPs a procesar según el límite
            ips_a_procesar = df.columns[:LIMIT_IPS] if LIMIT_IPS else df.columns
            num_ips = len(ips_a_procesar)
            
            print(f"[PROCESANDO] Etiqueta: {etiqueta:<15} ({num_ips} IPs) ... ", end="")
            sys.stdout.flush() 

            for ip in ips_a_procesar:
                serie = df[ip].values
                
                # 1. Generar gráficas DWT (Haar y Db2)
                for wav in WAVELETS_DWT:
                    ruta_final = os.path.join(OUTPUT_ROOT, wav, etiqueta)
                    os.makedirs(ruta_final, exist_ok=True)
                    try:
                        plot_mra_dwt(serie, wav, ip, etiqueta, ruta_final)
                    except: pass

                # 2. Generar gráficas CWT (Mexican Hat)
                ruta_mexh = os.path.join(OUTPUT_ROOT, "mexican_hat", etiqueta)
                os.makedirs(ruta_mexh, exist_ok=True)
                try:
                    plot_cwt_mexh(serie, ip, etiqueta, ruta_mexh)
                except: pass
            
            print("[COMPLETADO]")

        except Exception as e:
            print(f"\n[ERROR] Fallo en {etiqueta}: {e}")

    print("\n" + "="*50)
    print("PROCESO TERMINADO EXITOSAMENTE")

if __name__ == "__main__":
    generar_graficas()